In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# --- 1. CONFIGURAZIONE PERCORSI ---
PATH_INPUT = 'data_elaborated/train/train_cleaned.csv'
DIR_OUTPUT = 'data_elaborated/train/'
FILE_OUTPUT = 'train_with_residuals.csv'

# Variabili Operative (X) - Definiscono l'ambiente di volo
S_O = ['Sensed_Altitude', 'Sensed_Mach', 'Sensed_Pamb', 'Sensed_TAT', 'Sensed_Fan_Speed']

# Sensori di Degrado (y) - Quelli che vogliamo "pulire"
S_D = ['Sensed_T45', 'Sensed_T3', 'Sensed_WFuel', 'Sensed_Ps3', 'Sensed_Core_Speed']

def calculate_residuals(df):
    engine_results = []
    motori = df['ESN'].unique()
    
    print(f"Calcolo residui per {len(motori)} motori...")
    
    for esn in motori:
        # Isoliamo i dati del singolo motore
        engine_df = df[df['ESN'] == esn].copy()
        
        # Riordiniamo per sicurezza temporale
        engine_df = engine_df.sort_values('Cycles_Since_New')
        
        # Gestione valori mancanti (necessaria per la regressione)
        engine_df = engine_df.ffill().bfill()
        
        for sensor in S_D:
            # Addestriamo il modello "nominale" del motore
            X = engine_df[S_O].values
            y = engine_df[sensor].values
            
            # Regressione Lineare: impara il comportamento sano in base alla quota
            model = LinearRegression()
            model.fit(X, y)
            
            # Calcoliamo la differenza (Residuo)
            y_pred = model.predict(X)
            engine_df[f"{sensor}_res"] = y - y_pred
            
        engine_results.append(engine_df)
        print(f" > Motore {esn} completato.")
        
    return pd.concat(engine_results, ignore_index=True)

# --- 2. ESECUZIONE ---
if __name__ == "__main__":
    if os.path.exists(PATH_INPUT):
        print(f"Lettura dati puliti da: {PATH_INPUT}")
        df_cleaned = pd.read_csv(PATH_INPUT)
        
        # Calcolo
        df_final = calculate_residuals(df_cleaned)
        
        # Salvataggio
        if not os.path.exists(DIR_OUTPUT): os.makedirs(DIR_OUTPUT)
        df_final.to_csv(os.path.join(DIR_OUTPUT, FILE_OUTPUT), index=False)
        
        print("-" * 30)
        print(f"SUCCESSO: Creati residui per {len(S_D)} sensori.")
        print(f"File salvato: {os.path.join(DIR_OUTPUT, FILE_OUTPUT)}")
    else:
        print(f"ERRORE: File {PATH_INPUT} non trovato!")

Lettura dati puliti da: data_elaborated/train/train_cleaned.csv
Calcolo residui per 4 motori...
 > Motore 101 completato.
 > Motore 102 completato.
 > Motore 103 completato.
 > Motore 104 completato.
------------------------------
SUCCESSO: Creati residui per 5 sensori.
File salvato: data_elaborated/train/train_with_residuals.csv
